# 06 · final project

**before you start:** *Runtime → Change runtime type → T4 GPU → Save.* it's free, and it makes the AI models in this notebook about 20 times faster.

**how to use this:** press ▶ on each cell, top to bottom. anything in a box on the right is yours to change. change it, press ▶ again, see what happens. you can't break anything. if it gets weird: *Runtime → Restart session* and start from the top.

## the brief
teams of 2–3. pick **one** real problem, solve it with **one** of the four tracks below, and show it working.

no new code. every track is a form: you fill in the boxes, press ▶, and screenshot the result for your presentation.

*(you can also build your project in teachable machine. that counts.)*

## step 1 · plan it (5 minutes, on paper)
| | |
|---|---|
| **problem** | who has it, and why does it matter? |
| **input** | what goes in? a photo, a sentence, a row of data, a question |
| **ai** | which track below? |
| **output** | what comes out, and who acts on it? |
| **risk** | when could it be wrong, and what happens then? |

In [ ]:
#@title ▶ run me first (loads everything; takes a minute the first time)
import io, requests, numpy as np, pandas as pd, matplotlib.pyplot as plt, seaborn as sns, torch
from PIL import Image
from transformers import pipeline, AutoTokenizer, AutoModelForCausalLM
from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeClassifier, plot_tree
import warnings; warnings.filterwarnings("ignore")
device = "cuda" if torch.cuda.is_available() else "cpu"
CHAT_MODEL = "Qwen/Qwen2.5-1.5B-Instruct" if device == "cuda" else "Qwen/Qwen2.5-0.5B-Instruct"

def load_image(url_or_upload):
    if url_or_upload == "upload":
        from google.colab import files
        f = files.upload(); return Image.open(io.BytesIO(next(iter(f.values())))).convert("RGB")
    r = requests.get(url_or_upload, headers={"User-Agent": "Mozilla/5.0 (ai-bootcamp notebook)"}, timeout=30)
    r.raise_for_status(); img = Image.open(io.BytesIO(r.content)).convert("RGB"); img.thumbnail((1024, 1024)); return img

print("ready ✅")

---
## track A · an image sorter
examples: *ripe / unripe banana*, *healthy / sick leaf*, *recyclable / not recyclable*, *helmet / no helmet*.

In [ ]:
#@title A · sort photos into your labels
problem = "is this fruit ready to eat?" #@param {type:"string"}
labels = "a ripe banana, an unripe green banana, a rotten banana" #@param {type:"string"}
photo_links = "https://commons.wikimedia.org/wiki/Special:FilePath/Bananas_white_background_DS.jpg?width=800 | https://commons.wikimedia.org/wiki/Special:FilePath/Unripe_Green_Bananas.jpg?width=800" #@param {type:"string"}
matcher = pipeline("zero-shot-image-classification", model="openai/clip-vit-base-patch32")
label_list = [l.strip() for l in labels.split(",")]
links = [p.strip() for p in photo_links.split("|")]
fig, axes = plt.subplots(1, len(links), figsize=(6 * len(links), 6), squeeze=False)
for ax, link in zip(axes[0], links):
    img = load_image(link); best = matcher(img, candidate_labels=label_list)[0]
    ax.imshow(img); ax.axis("off"); ax.set_title(f"{best['label']}  ({best['score']:.0%})")
fig.suptitle(problem, fontsize=16); plt.show()

separate photo links with `|`, or type `upload`.

---
## track B · a text sorter
examples: *sort student feedback*, *flag urgent complaints*, *tag news headlines by topic*.

In [ ]:
#@title B · sort messages into your boxes
problem = "which complaints need a human today?" #@param {type:"string"}
boxes = "urgent, refund request, delivery delay, just feedback" #@param {type:"string"}
messages = "my gas cylinder is leaking, please send someone | i want my money back for the broken fan | order still not here after a week | the new app design is nice" #@param {type:"string"}
sorter = pipeline("zero-shot-classification", model="facebook/bart-large-mnli")
box_list = [b.strip() for b in boxes.split(",")]
rows = []
for m in [m.strip() for m in messages.split("|")]:
    r = sorter(m, candidate_labels=box_list)
    rows.append({"message": m, "box": r["labels"][0], "sure": f"{r['scores'][0]:.0%}"})
print(problem); pd.DataFrame(rows)

look at the last message: none of your boxes fit a compliment, so it **picked one anyway**. add a box like `compliment` and run it again.

---
## track C · a predictor
pick a real dataset and the column you want to predict. the computer grows a decision tree and tells you how well it does on rows it never saw.

In [ ]:
#@title C · predict a column
dataset = "penguins" #@param ["titanic", "penguins", "tips", "diamonds"]
predict = "species" #@param {type:"string"}
depth = 3 #@param {type:"slider", min:1, max:6, step:1}
df = sns.load_dataset(dataset).dropna()
if predict not in df.columns:
    raise SystemExit(f"'{predict}' isn't a column. pick one of: {', '.join(df.columns)}")
y = df[predict].astype(str)
X = pd.get_dummies(df.drop(columns=[predict]), drop_first=True).astype(float)
X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.25, random_state=0)
tree = DecisionTreeClassifier(max_depth=depth, random_state=0).fit(X_tr, y_tr)
lazy = (y_te == y_tr.mode()[0]).mean()
print(f"{len(df)} rows. predicting '{predict}' from {X.shape[1]} other columns.")
print(f"lazy guess (always '{y_tr.mode()[0]}'): {lazy:.0%}   ·   your tree: {tree.score(X_te, y_te):.0%}")
plt.figure(figsize=(18, 7)); plot_tree(tree, feature_names=list(X.columns), class_names=sorted(y.unique()) if y.nunique() < 8 else None, filled=True, rounded=True, impurity=False, fontsize=9); plt.show()

columns you can predict: **titanic** → `survived`, `class`, `who` · **penguins** → `species`, `island`, `sex` · **tips** → `smoker`, `day`, `time` · **diamonds** → `cut`, `color`.

for your presentation: *what did the tree ask first?* that's the single most useful clue in the data.

---
## track D · a chatbot with a job
examples: *a campus guide*, *a polite complaint handler*, *a quiz master for your subject*.

In [ ]:
#@title D · write its job, then test it
its_job = "You are the help desk for a college canteen. You only answer questions about the menu, prices and timings. Menu: chai 15, coffee 20, samosa 20, vada pav 25, veg biryani 90. Open 8am to 6pm, closed Sundays. If you don't know, say so." #@param {type:"string"}
question_1 = "how much for two chais and a samosa?" #@param {type:"string"}
question_2 = "are you open on sunday?" #@param {type:"string"}
question_3 = "who won the cricket match yesterday?" #@param {type:"string"}
tok = AutoTokenizer.from_pretrained(CHAT_MODEL)
lm = AutoModelForCausalLM.from_pretrained(CHAT_MODEL, torch_dtype=torch.float16 if device == "cuda" else torch.float32).to(device)
for q in [question_1, question_2, question_3]:
    chat = [{"role": "system", "content": its_job}, {"role": "user", "content": q}]
    inputs = tok.apply_chat_template(chat, add_generation_prompt=True, return_tensors="pt", return_dict=True).to(device)
    with torch.no_grad():
        out = lm.generate(**inputs, max_new_tokens=100, do_sample=False)
    print(f"🧑 {q}\n🤖 {tok.decode(out[0][inputs['input_ids'].shape[1]:], skip_special_tokens=True).strip()}\n")

question 3 is the trap: a good help desk says *"i don't know"*. does yours? check its maths on question 1 too.

---
## step 2 · test it properly
- try **3 inputs it should get right** and **2 it should get wrong** on purpose.
- write down one time it was confidently wrong. that's the most interesting slide.

## step 3 · present it (3 minutes per team)
1. **the problem**, in one sentence
2. **input → ai → output**, as three boxes on a slide
3. **a live demo**, or screenshots
4. **one failure**, and why you think it happened
5. **what you'd fix with better data**

<small>photos: bananas by Augustus Binu (CC BY-SA 3.0) and Nabunje Leticia (CC BY-SA 4.0), via Wikimedia Commons.</small>